# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [23]:
from dataclasses import dataclass
import csv
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

../../../../../../assets/03-wine-quality/inference_samples.csv
True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [24]:
# TODO: implementa la lectura del CSV.
# Pista: usa DATA_PATH.open(newline="", encoding="utf-8")
rows = []
row = {}

with DATA_PATH.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for line in reader:
        rows.append(line)

row = rows[0]

print("Filas:", len(rows))
print("Columnas:", list(row.keys()))
print("Primera fila:", row)

Filas: 5
Columnas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Primera fila: {'sample_id': 'red-001', 'fixed_acidity': '7.4', 'volatile_acidity': '0.7', 'citric_acid': '0.0', 'residual_sugar': '1.9', 'chlorides': '0.076', 'free_sulfur_dioxide': '11.0', 'total_sulfur_dioxide': '34.0', 'density': '0.9978', 'ph': '3.51', 'sulphates': '0.56', 'alcohol': '9.4'}


### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo?
   sample_id. Identifica la muestra, pero no entra en el vector del modelo.

2. ¿Cuántas características numéricas hay?
   11 características numéricas.

3. ¿Qué columna extra añadirías para probar el contrato?
   unexpected_field. El contrato debe rechazarla porque no está declarada (extra="forbid").

4. ¿Qué valor de `ph` sería claramente inválido?
   ph = 99. Está fuera del rango 2.5–4.5 (y un pH real va de 0 a 14).

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Rango orientativo |
|---|---|
| `fixed_acidity` | 0–20 |
| `volatile_acidity` | 0–2 |
| `citric_acid` | 0–2 |
| `residual_sugar` | 0–20 |
| `chlorides` | 0–1 |
| `free_sulfur_dioxide` | 0–100 |
| `total_sulfur_dioxide` | 0–300 |
| `density` | 0.98–1.01 |
| `ph` | 2.5–4.5 |
| `sulphates` | 0–3 |
| `alcohol` | 5–20 |

In [25]:
class WineQualityRequestPydantic(BaseModel):
    # TODO: prohíbe campos extra con model_config.
    model_config = ConfigDict(extra="forbid")

    # TODO: declara las 11 características con Field(ge=..., le=...).
    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field(ge=0, le=20)
    chlorides: float = Field(ge=0, le=1)
    free_sulfur_dioxide: float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)

# TODO: separa sample_id de la fila y valida solo las características del modelo.
model_row = dict(row)
del model_row["sample_id"]
request = WineQualityRequestPydantic.model_validate(model_row)
print(request)

fixed_acidity=7.4 volatile_acidity=0.7 citric_acid=0.0 residual_sugar=1.9 chlorides=0.076 free_sulfur_dioxide=11.0 total_sulfur_dioxide=34.0 density=0.9978 ph=3.51 sulphates=0.56 alcohol=9.4


### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

In [26]:
# TODO: completa los casos y captura ValidationError.


valid_row = dict(model_row)

extra_row = dict(model_row)
extra_row["unexpected_field"] = "1"

bad_ph_row = dict(model_row)
bad_ph_row["ph"] = "99"

cases = [("fila válida", valid_row), ("columna extra", extra_row), ("ph fuera de rango", bad_ph_row)]

for name, data in cases:
    try:
        WineQualityRequestPydantic.model_validate(data)
        print(name, "-> OK")
    except ValidationError as error:
        print(name, "-> ERROR")
        print(error)

fila válida -> OK
columna extra -> ERROR
1 validation error for WineQualityRequestPydantic
unexpected_field
  Extra inputs are not permitted [type=extra_forbidden, input_value='1', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
ph fuera de rango -> ERROR
1 validation error for WineQualityRequestPydantic
ph
  Input should be less than or equal to 4.5 [type=less_than_equal, input_value='99', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [27]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    # TODO: declara las mismas 11 características.
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float

# TODO: crea una instancia válida y prueba qué ocurre con ph=99.0.
valid_values = request.model_dump()
valid = WineQualityRequestDataclass(**valid_values)
print(valid)

bad_values = request.model_dump()
bad_values["ph"] = 99.0
bad = WineQualityRequestDataclass(**bad_values)
print(bad)

text = WineQualityRequestDataclass(**model_row)
print(type(text.ph))

WineQualityRequestDataclass(fixed_acidity=7.4, volatile_acidity=0.7, citric_acid=0.0, residual_sugar=1.9, chlorides=0.076, free_sulfur_dioxide=11.0, total_sulfur_dioxide=34.0, density=0.9978, ph=3.51, sulphates=0.56, alcohol=9.4)
WineQualityRequestDataclass(fixed_acidity=7.4, volatile_acidity=0.7, citric_acid=0.0, residual_sugar=1.9, chlorides=0.076, free_sulfur_dioxide=11.0, total_sulfur_dioxide=34.0, density=0.9978, ph=99.0, sulphates=0.56, alcohol=9.4)
<class 'str'>


### Decisión de diseño

Completa la frase:

> Usaría Pydantic en ___en la frontera de entrada___ porque __pueden venir mal los datos externos__. Usaría `dataclass` en _ datos internos ya validados___ porque __es mas ligero y no necesita volver a avlidar___.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [28]:
# TODO: rellena el orden exacto a partir del CSV y del contrato.
FEATURE_NAMES = (
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
)

# TODO: construye vector usando getattr(request, name).
vector = []
for name in FEATURE_NAMES:
    vector.append(getattr(request, name))

print(len(vector))
print(vector)

11
[7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]


### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores?
  Sí. Sigue teniendo 11 números y el mismo tamaño, así que no salta ningún error de forma.

- ¿El clasificador detectaría necesariamente el error?
  No. El modelo solo recibe números por posición: recibiría 9.4 donde espera la densidad (≈ 0.99) y 0.9978 donde espera el alcohol. Predice igualmente, pero mal, y sin avisar. Es un error silencioso.

- ¿Cómo evitarías que cada script definiera un orden distinto?
  Definiendo FEATURE_NAMES una sola vez, en preprocess.py, e importándolo desde el resto de scripts. Además, al cargar el modelo se compara su feature_names con FEATURE_NAMES y, si no coinciden, se lanza un ValueError.

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [29]:
from typing import Literal
class WineQualityPredictionPydantic(BaseModel):
    # TODO: declara los campos de salida y sus restricciones.
    model_config = ConfigDict(extra="forbid")

    sample_id: str = Field(min_length=1)
    quality_band: Literal["needs_review", "acceptable", "excellent"]
    confidence: float = Field(ge=0, le=1)
    model_version: str = Field(min_length=1)
    preprocessing_version: str = Field(min_length=1)

# TODO: escribe un ejemplo de salida para red-001.
example = WineQualityPredictionPydantic(
    sample_id="red-001",
    quality_band="acceptable",
    confidence=0.82,
    model_version="wine-quality-v1",
    preprocessing_version="wine-red-features-v1",
)
print(example)

sample_id='red-001' quality_band='acceptable' confidence=0.82 model_version='wine-quality-v1' preprocessing_version='wine-red-features-v1'


## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.


CSV → contrato → preprocesado → modelo cargado → predicción

1. CSV: csv.DictReader lee cada fila como un diccionario de textos.
2. Contrato (WineQualityRequest): se separa sample_id, se convierten los tipos y se comprueban los rangos. Una columna extra o un valor fuera de rango se rechazan aquí.
3. Preprocesado: se construye el vector de 11 números en el orden de FEATURE_NAMES.
4. Modelo cargado: se carga el .joblib y se comprueba que su feature_names coincide con FEATURE_NAMES.
5. Predicción: predict / predict_proba y se valida la salida con WineQualityPrediction (categoría, confianza y versiones).

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.

## Entrega

### Tabla de contrato

| # | Campo | Tipo | Unidad | Rango | ¿Admite ausentes? |
|---|---|---|---|---|---|
| 1 | fixed_acidity | float | g/L (ácido tartárico) | 0–20 | No |
| 2 | volatile_acidity | float | g/L (ácido acético) | 0–2 | No |
| 3 | citric_acid | float | g/L | 0–2 | No |
| 4 | residual_sugar | float | g/L | 0–20 | No |
| 5 | chlorides | float | g/L (cloruro sódico) | 0–1 | No |
| 6 | free_sulfur_dioxide | float | mg/L | 0–100 | No |
| 7 | total_sulfur_dioxide | float | mg/L | 0–300 | No |
| 8 | density | float | g/cm³ | 0.98–1.01 | No |
| 9 | ph | float | sin unidad | 2.5–4.5 | No |
| 10 | sulphates | float | g/L (sulfato potásico) | 0–3 | No |
| 11 | alcohol | float | % vol | 5–20 | No |

sample_id (texto, no vacío) identifica la muestra y no entra en el vector. Se rechaza cualquier columna extra (extra="forbid").

### Clasificación de las piezas
- Datos históricos de entrenamiento: los usó el docente para entrenar. No los recibimos y no entrenamos nada.
- Artefacto entregado (.joblib): el modelo ya entrenado, con estimator, feature_names y model_version.
- Datos nuevos de inferencia (inference_samples.csv): 5 muestras sin etiqueta de calidad. Pasan por el contrato antes de llegar al modelo.

### Orden del vector
fixed_acidity, volatile_acidity, citric_acid, residual_sugar, chlorides, free_sulfur_dioxide, total_sulfur_dioxide, density, ph, sulphates, alcohol

### Casos de fallo y etapa de rechazo
| Caso | Ejemplo | Etapa | Razón |
|---|---|---|---|
| Fila válida | red-001 tal cual | No falla | Cumple tipos y rangos |
| Columna desconocida | unexpected_field=1 | Contrato de entrada | No está declarada (extra="forbid") |
| Fuera de rango | ph=99 | Contrato de entrada | Supera el máximo de 4.5 |
| Artefacto incompatible | feature_names distinto | Carga del modelo | El orden no coincide: ValueError |

### Contrato de salida
sample_id, quality_band (needs_review / acceptable / excellent), confidence (0–1), model_version, preprocessing_version.

### Criterios de aceptación (para la clase 2)
1. uv run pytest termina sin fallos y el vector tiene 11 valores en el orden de FEATURE_NAMES, sin sample_id.
2. Una fila inválida (columna extra o ph fuera de rango) para el proceso y no deja un CSV de salida parcial.
3. El comando uv run python -m model_inference.predict_file --input assets/inference_samples.csv --output .tmp/wine_predictions.csv escribe 5 filas con sample_id, quality_band, confidence, model_version y preprocessing_version.

### Plan de implementación
- contracts.py: WineQualityRequest y WineQualityPrediction con Pydantic.
- preprocess.py: FEATURE_NAMES, PREPROCESSING_VERSION y la función que crea el vector. No lee ficheros.
- inference.py: carga el .joblib, comprueba feature_names y que el estimador tenga predict / predict_proba, predice y valida la salida.
- predict_file.py: CLI que lee el CSV, valida todas las filas, predice y escribe la salida solo si todo ha ido bien.

